# 📝 Notebook 2 — Create the Evaluation Dataset
## RAGAS Evaluation Series · Part 2 of 4

---

## 🎯 What this notebook does

RAGAS can compute most metrics with just **question + answer + contexts**.  
But for **Context Recall**, it also needs a **ground truth** — the correct answer written by a human.

This notebook creates the evaluation dataset: a list of questions paired with their human-verified ground truths.

```
evaluation_data = [
    {
        "question":     "What is the maternity leave policy?",
        "ground_truth": "Employees are eligible for 6 months maternity leave."
    },
    ...
]
```

> 💡 **Think of ground truth as the answer key.**  
> Just like an exam marking scheme, it lets RAGAS check whether the retriever  
> found ALL the information needed — not just some of it.

---

## 🔑 Why ground truth matters for Context Recall

```
Question: "What is the maternity leave policy?"

Ground truth: "Employees are eligible for 6 months maternity leave."

Retriever returns:
  Chunk 1: "Employees are eligible for 6 months maternity leave."  ← covers it ✅
  Chunk 2: "Annual leave balance is 24 days."
  Chunk 3: "Work from home is allowed twice a week."

Context Recall = 1.0  (ground truth is fully covered by retrieved chunks)
```

```
Question: "What benefits are available?"

Ground truth: "Employees get maternity leave, annual leave, medical insurance, and a performance bonus."

Retriever returns:
  Chunk 1: "Annual leave balance is 24 days."
  Chunk 2: "Work from home is allowed twice a week."
  ← MISSING: maternity leave, medical insurance, performance bonus

Context Recall = Low  (retriever missed most of the answer)
```

---

## 📋 What makes a good ground truth?

| Rule | Good example | Bad example |
|---|---|---|
| Exact document language | "Employees are eligible for 6 months maternity leave" | "6 months leave" |
| Complete answer | "60 days notice period before resignation" | "60 days" |
| One fact per entry | Keep it focused | Don't combine 5 facts into one |
| No paraphrasing | Copy from the document | Don't rephrase |

In [1]:
import json

print("✅ Ready to create evaluation dataset")

✅ Ready to create evaluation dataset


---
## 📄 Step 1 — Write Your Evaluation Dataset

Replace or extend the questions and ground truths below to match your own `company_policy.txt`.

> ⚠️ **Ground truths must come from your actual document.**  
> Copy the relevant sentence directly — RAGAS uses exact text matching internally.

In [2]:
# ── Evaluation dataset ────────────────────────────────────────────────────────
#
# Format:
#   question:     what you ask the RAG system
#   ground_truth: the correct, complete answer taken directly from the document
#
# This is your "answer key" — the source of truth RAGAS compares against.

evaluation_data = [
    {
        "question":     "What is the maternity leave policy?",
        "ground_truth": "Employees are eligible for 6 months maternity leave."
    },
    {
        "question":     "How many days annual leave do employees get?",
        "ground_truth": "Annual leave balance is 24 days."
    },
    {
        "question":     "What is the notice period before resignation?",
        "ground_truth": "Employees must serve 60 days notice period before resignation."
    },
    {
        "question":     "Can employees work from home?",
        "ground_truth": "Work from home is allowed twice a week."
    },
    {
        "question":     "What is the probation period for new employees?",
        "ground_truth": "Probation period for new employees is 3 months."
    },
]

print(f"✅ Evaluation dataset created — {len(evaluation_data)} entries")
print()
for i, entry in enumerate(evaluation_data, 1):
    print(f"  [{i}] Q: {entry['question']}")
    print(f"       A: {entry['ground_truth']}")
    print()

✅ Evaluation dataset created — 5 entries

  [1] Q: What is the maternity leave policy?
       A: Employees are eligible for 6 months maternity leave.

  [2] Q: How many days annual leave do employees get?
       A: Annual leave balance is 24 days.

  [3] Q: What is the notice period before resignation?
       A: Employees must serve 60 days notice period before resignation.

  [4] Q: Can employees work from home?
       A: Work from home is allowed twice a week.

  [5] Q: What is the probation period for new employees?
       A: Probation period for new employees is 3 months.



---
## 💾 Step 2 — Save to File

We save this as `evaluation_dataset.json` so Notebook 4 can load it alongside the RAG outputs.

In [3]:
# ── Save evaluation dataset ───────────────────────────────────────────────────
with open("evaluation_dataset.json", "w") as f:
    json.dump(evaluation_data, f, indent=2)

print("💾 Saved evaluation_dataset.json")
print()
print("📋 File contents preview:")
print(json.dumps(evaluation_data[:2], indent=2))
print()
print("➡️  Next: open Notebook 3 to run RAG and capture outputs.")

💾 Saved evaluation_dataset.json

📋 File contents preview:
[
  {
    "question": "What is the maternity leave policy?",
    "ground_truth": "Employees are eligible for 6 months maternity leave."
  },
  {
    "question": "How many days annual leave do employees get?",
    "ground_truth": "Annual leave balance is 24 days."
  }
]

➡️  Next: open Notebook 3 to run RAG and capture outputs.


---
## 🔬 Bonus — What if you don't have ground truths?

RAGAS can still run without ground truths — it just skips Context Recall.

| Metric | Needs ground truth? |
|---|---|
| Faithfulness | No |
| Answer Relevancy | No |
| Context Precision | No |
| Context Recall | Yes ✅ |

> 💡 **Production tip:** For large document sets, use the LLM itself to generate  
> candidate ground truths from the document, then have a human reviewer verify them.  
> This is called **synthetic evaluation data generation** and RAGAS has built-in support for it.